# fastcharts — demo

Millions of points, interactive, in a notebook. Pan by dragging, zoom with the wheel, double-click to reset.
Shift-drag a scatter to box-select. Two APIs over one engine: the fluent `Figure` and the Reflex-flavored `fc.scatter_chart(...)`.

In [ ]:
import numpy as np

import fastcharts as fc
import fastcharts.kernels as k
from fastcharts import Figure

rng = np.random.default_rng(0)
print("kernel backend:", k.BACKEND)

## 10M-point line (ships decimated: ≤4 points per pixel column)

In [ ]:
n = 10_000_000
t = np.datetime64("2015-01-01") + np.arange(n).astype("timedelta64[s]")
y = np.cumsum(rng.normal(size=n))
y[n // 2] += 80  # a single-sample spike M4 must preserve

fc.line_chart(
    fc.line(x=t, y=y, name="random walk"),
    fc.x_axis(label="time"), fc.y_axis(label="value"),
    title="10M points — wheel to zoom into the spike", width=950, height=420,
)

## Composition API (Reflex-flavored) + per-point color/size + events
Hover a point for its exact f64 row; shift-drag to box-select (unselected marks dim).

In [ ]:
m = 20_000
xs = rng.normal(0, 1, m)
ys = xs * 0.5 + rng.normal(0, 0.6, m)
depth = xs**2 + ys**2                  # continuous -> viridis colormap
weight = np.abs(rng.normal(size=m))    # continuous -> size 3..18 px

fc.scatter_chart(
    fc.scatter(x=xs, y=ys, color=depth, size=weight, colormap="viridis", size_range=(3, 18), opacity=0.7),
    fc.x_axis(label="x"), fc.y_axis(label="y"),
    title="colored + sized scatter", width=950, height=420,
    on_select=lambda sel: print(len(sel), "points selected"),
)

## data= + column names (the Reflex data_key idiom), categorical palette

In [ ]:
df = {
    "x": xs, "y": ys,
    "grp": np.array(["alpha", "beta", "gamma"])[rng.integers(0, 3, m)],
}
fc.scatter_chart(
    fc.scatter(x="x", y="y", color="grp", data=df, size=4.0, opacity=0.6),
    title="categorical scatter (color='grp')", width=950, height=420,
)

## 30M-point scatter -> Tier-2 density surface (§5)
Above ~200k points scatter auto-aggregates: the kernel bins the viewport into a
count grid, the client colormaps it on the GPU, and zoom re-bins the visible window.

In [ ]:
big = 30_000_000
bx = rng.normal(0, 1, big)
by = bx * 0.7 + rng.normal(0, 0.5, big)

chart = fc.scatter_chart(
    fc.scatter(x=bx, y=by, colormap="magma"),
    title="30M points (density)", width=950, height=420,
)
print("tier:", chart.figure().build_payload()[0]["traces"][0]["tier"])
chart

## The memory story (§27: if it isn't in the report, it isn't real)

In [ ]:
fig = Figure().line(t, y)
report = fig.memory_report()
print(f"canonical: {report['canonical_bytes'] / 2**20:.1f} MB kernel-side")
print(f"first-paint transport: {report['transport_bytes_first_paint'] / 2**10:.1f} KB")
print(f"transport bytes/point: {report['transport_bytes_per_point']:.4f}")

## Standalone HTML export (no kernel needed to view; hover + select still work)

In [ ]:
fc.scatter_chart(fc.scatter(x=xs, y=ys, color=depth)).to_html("scatter_chart.html")
print("wrote scatter_chart.html")